<b><font color='aquamarine' size="6"> Data Preprocessing and Visualization

<b><font size="6" color="aquamarine">City Hospital – Data Preprocessing &
Exploratory Analysis</i></font></b><a class="anchor"></a><a id="toc"></a><br>

### Group 16 - Team Members

| Name | Student Number |
|---|---|
| João Umbelino | 20251309 |
| Mariana Monteiri | 20251290 |
| Martim Violante | 20251310 |
| Rafael Portela | 20251361 |


---
## <font color='#BFD72F'>Table of Contents</font> <a class="anchor" id='toc'></a>
- [1. Part A - Data Understanding & Cleaning](#1)
  - [1.1 Loading and Importing the Data](#1_1)
    - [1.1.1 Initial Understanding of the Dataset](#1_1_1)
  - [1.2 Handling Duplicate Records](#1_2)
    - [1.2.1 Identifying Duplicate Rows](#1_2_1)
    - [1.2.2 Removing Duplicate Records](#1_2_2)
    - [1.2.3 Justifying the Cleaning Decisions](#1_2_3)
  - [1.3 Identifying and Handling Missing Values](#1_3)
    - [1.3.1 Identifying Missing Values Across Variables](#1_3_1)
    - [1.3.2 Analysing Missing Data Patterns](#1_3_2)
    - [1.3.3 Applying Imputation Strategies](#1_3_3)
  - [1.4 Handle Inconsistent Data](#1_4)
    - [1.4.1 Identifying Inconsistent Data (Negative Values, Incorrect Dates)](#1_4_1)
    - [1.4.2 Correcting Data Types for Optimal Performance](#1_4_2)
    - [1.4.3 Checking Demographic and Socio-Economic Variables](#1_4_3)
    - [1.4.4 Identifying Inconsistent Patient Attributes (Static Features)](#1_4_4)
    - [1.4.5 Correcting Inconsistencies Across Patient Visits](#1_4_5)
    - [1.4.6 Identifying Invalid Category Labels](#1_4_6)
    - [1.4.7 Correcting Satisfaction Level Values](#1_4_7)
    - [1.4.8 Checking Logical Contradictions](#1_4_8)
    - [1.4.9 Checking Invalid Insurance Coverages](#1_4_9)
  - [1.5 Identifying and Handling Outliers](#1_5)
    - [1.5.1 Detecting Outliers in Numerical Variables](#1_5_1)
    - [1.5.2 Visualizing Potential Outliers](#1_5_2)
    - [1.5.3 Deciding How to Handle Outliers](#1_5_3)

- [2. Part B - Feature Engineering & ABT Construction](#2)
  - [2.1 Patient-Level ABT Construction](#2_1)
    - [2.1.1 Static Attributes](#2_1_1)
    - [2.1.2 Visit Behaviour Features](#2_1_2)
    - [2.1.3 Financial Features](#2_1_3)
    - [2.1.4 Satisfaction Features](#2_1_4)
    - [2.1.5 Recency and Activity Features](#2_1_5)
  - [2.2 Data Dictionary](#2_2)
  - [2.3 Exporting the ABT](#2_3)

- [3. Part C - Exploratory Visualization & Insights](#3)
  - [3.1 Patient Demographics](#3_1)
    - [3.1.1 Age Distribution](#3_1_1)
    - [3.1.2 Gender Distribution](#3_1_2)
    - [3.1.3 Income Distribution](#3_1_3)
  - [3.2 Department and Service Usage](#3_2)
    - [3.2.1 Consultations by Department](#3_2_1)
    - [3.2.2 Consultations Over Time](#3_2_2)
    - [3.2.3 Department Usage by Gender or Age Group](#3_2_3)
  - [3.3 Patient Satisfaction](#3_3)
    - [3.3.1 Satisfaction by Department](#3_3_1)
    - [3.3.2 Satisfaction by Patient Segment](#3_3_2)
  - [3.4 Additional Visualizations](#3_4)
  - [3.5 Management Questions and Key Insights](#3_5)

### <font color='turquoise' size='6'>1. Part A – Data Understanding & Cleaning </font> <a class="anchor" id="1"></a>
  [Back to TOC](#toc)

Every issue is detected in code, a decision is taken (**correct / impute / flag / remove**) and explained in the text right below it. Patients are never removed: every problem is repaired or flagged at visit level.


In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## <font color='turquoise'>1.1 Loading and Importing the Data

In [35]:
# keep_default_na=False: the text "None" (= no insurance / no family history) must not become NaN
data = pd.read_excel("Project_Data.xlsx", keep_default_na=False, na_values=[""])

print(f"Dimensions      : {data.shape[0]:,} rows x {data.shape[1]} columns")
print(f"Unique patients : {data['Patient ID'].nunique()}")
print("\nData types:")
print(data.dtypes.to_string())

df = data.copy()               # all cleaning happens on this copy

FileNotFoundError: [Errno 2] No such file or directory: 'Project_Data.xlsx'

## <font color='turquoise'>1.1.1 Initial Understanding of the Dataset

In [ ]:
display(data.head())

print("Visit dates      :", data["Visit Date"].min().date(), "->", data["Visit Date"].max().date())
rpp = data.groupby("Patient ID").size()
print("Visits / patient :", rpp.min(), "to", rpp.max(), f"(mean {rpp.mean():.1f})")

print("\nNumerical summary:")
display(data.describe().T.round(2))

print("\nNumber of distinct labels in the categorical variables:")
cat_raw = data.select_dtypes(exclude=["number", "datetime"]).columns
display(data[cat_raw].nunique().to_frame("n_labels").T)

**First impressions.** The file has 16 variables, 10,008 consultations and 500 patients (9-39 visits each). Dates cover Jan-Jun 2024. Already visible in the summary: a maximum age of 195, a maximum price of ~£40,000, a maximum income of ~£12M, durations up to 600 minutes and a satisfaction maximum of 6 on a 1-5 scale. Several variables are missing values, and some patient attributes (gender, marital status, profession) may change between visits. Each of these is examined below.

Note: the text `"None"` (no insurance / no family history) is a valid category, not a missing value, so the file is read with `keep_default_na=False`; otherwise pandas would silently turn it into NaN.

## <font color='turquoise'>1.2 Checking Duplicate Records

## <font color='turquoise'>1.2.1 Identifying Duplicate Rows

In [ ]:
# (a) exact duplicates: all 16 fields identical
dup_exact = df.duplicated()
print("Exact duplicate rows (all 16 fields identical):", int(dup_exact.sum()))
display(df[df.duplicated(keep=False)].sort_values(["Patient ID", "Visit Date", "Department"]))

# (b) same patient + date + department, but at least one other field differs
key = ["Patient ID", "Visit Date", "Department"]
after_exact = df[~dup_exact]
print(f"\nRows sharing patient + date + department after removing exact duplicates: {int(after_exact.duplicated(key, keep=False).sum())}")
display(after_exact[after_exact.duplicated(key, keep=False)].sort_values(key).head(6))

## <font color='turquoise'>1.2.2 Removing Duplicate Records

In [ ]:
# (a) exact duplicates -> remove (keep first)
df = df.drop_duplicates().copy()

# (b) same patient/date/department but different values -> keep and flag
df["flag_same_day_same_dept"] = df.duplicated(key, keep=False)

print(f"Rows after removing exact duplicates: {len(df):,}  |  patients: {df['Patient ID'].nunique()}")
print(f"Fewest visits left for any patient: {df.groupby('Patient ID').size().min()}")
print(f"Rows flagged as same patient/date/department: {int(df['flag_same_day_same_dept'].sum())}")

## <font color='turquoise'> 1.2.3 Justifying the Cleaning Decisions

| Finding | Decision | Why |
|---|---|---|
| 8 rows identical in all 16 fields | **Remove** (keep first) | The price is a continuous number, so two visits matching to many decimals are a double-entry, not two consultations. |
| 114 rows share patient + date + department but differ in duration / rating / price | **Keep and flag** | The values differ, so they can be separate consultations on the same day. Deleting them would remove real activity; the flag lets the next team exclude them if needed. |
| Patients lost | **None** | Every patient keeps at least 9 visits after de-duplication. |

## <font color='turquoise'>1.3 Identifying and Handling Missing Values

## <font color='turquoise'>1.3.1 Identifying Missing Values Across Variables

In [ ]:
data_cols = [c for c in df.columns if not c.startswith("flag_")]
mt = pd.DataFrame({"n_missing": df[data_cols].isna().sum(),
                   "pct_missing": (df[data_cols].isna().mean() * 100).round(2)})
display(mt[mt.n_missing > 0].sort_values("n_missing", ascending=False))
print("Rows with at least one missing value:", int(df[data_cols].isna().any(axis=1).sum()),
      "| rows with more than one:", int((df[data_cols].isna().sum(axis=1) > 1).sum()))

# hidden missing values: placeholder strings disguised as valid labels
placeholders = {"", " ", "?", "na", "n/a", "nan", "null", "unknown", "-", "--"}
obj_cols = df[data_cols].select_dtypes(exclude=["number", "datetime"]).columns
hidden = {c: int(df[c].dropna().astype(str).str.strip().str.lower().isin(placeholders).sum()) for c in obj_cols}
print("\nPlaceholder strings hiding missing values:", {k: v for k, v in hidden.items() if v} or "none found")

Five variables have genuine gaps: **Age**, **Education Level**, **Insurance Provider**, **Insurance Coverage** and **Approximate Annual Income**. No placeholder strings hide further missing data, and the label `"None"` is a legitimate category (no insurance / no family history), not a gap.

## <font color='turquoise'>1.3.2 Analysing Missing Data Patterns

In [ ]:
miss_cols = [c for c in data_cols if df[c].isna().any()]
share_missing = df[miss_cols].isna().groupby(df["Patient ID"]).mean()      # share of each patient's visits that are missing
display(pd.DataFrame({
    "rows_missing": df[miss_cols].isna().sum(),
    "patients_affected": (share_missing > 0).sum(),
    "patients_with_all_visits_missing": (share_missing == 1).sum(),
    "max_share_of_one_patient_visits": share_missing.max().round(2)}))

# does missingness relate to the department that recorded the visit?
mr = (df[miss_cols].isna().groupby(df["Department"]).mean() * 100).round(1)
display(mr)

fig, ax = plt.subplots(figsize=(8, 4.8))
sns.heatmap(mr, annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "% missing"}, ax=ax)
ax.set_title("Missing values by department (% of visits)"); ax.set_xlabel("Variable"); ax.set_ylabel("Department")
plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()

# do the gaps coincide in the same rows?
print("Correlation of missingness indicators (off-diagonal values near 0 = independent gaps):")
display(df[miss_cols].isna().corr().round(2))

**Reading the pattern.**
- Gaps are **sparse** (0.3%-1.5% of rows per variable) and almost never overlap: the missingness indicators are uncorrelated, so no visit is "mostly empty".
- Missingness depends on the **recording department**, not on the patient: Age is only missing in five departments, income in six others. This points to a data-capture issue in the source systems, so the true value is not related to the patient's characteristics (consistent with missing completely at random).
- Age, Education and Income are **per-patient attributes** that are present on the patient's other visits. Insurance Provider is entirely missing for just 2 patients, but the coverage amount on each visit identifies the provider (see below).

This justifies imputing from the patient's own other visits rather than from group averages.

## <font color='turquoise'>1.3.3 Applying Imputation Strategies

In [ ]:
def fill_from_patient(col):
    """Fill gaps of a static attribute with the same patient's most frequent value."""
    return df.groupby("Patient ID")[col].transform(lambda s: s.fillna(s.mode().iloc[0]) if s.notna().any() else s)


# --- Age and Education: static -> fill from the same patient's other visits -------------------------------
for col in ["Age", "Education Level"]:
    df[col] = fill_from_patient(col)

# --- Insurance provider: every provider pays a fixed share of the price -> infer it from coverage / price ---
RATE = {"None": 0.0, "Provider A": 0.8, "Provider B": 0.9, "Provider C": 0.7, "Provider D": 1.0}

ok = df["Insurance Provider"].notna() & df["Insurance Coverage"].notna() & (df["Consultation Price"] <= 1000)
print("Evidence: coverage / price by provider (min = median = the rate; larger maxima are the off-rule rows of 1.4.9)")
display((df.loc[ok, "Insurance Coverage"] / df.loc[ok, "Consultation Price"])
        .groupby(df.loc[ok, "Insurance Provider"]).agg(["median", "min", "max", "count"]).round(3))

ratio = df["Insurance Coverage"] / df["Consultation Price"]
usable = df["Insurance Provider"].isna() & ratio.notna() & (df["Consultation Price"] <= 1000)
df.loc[usable, "Insurance Provider"] = ratio[usable].map(lambda r: min(RATE, key=lambda k: abs(RATE[k] - r)))
df["Insurance Provider"] = fill_from_patient("Insurance Provider")      # fallback: patient's other visits

# --- Income stays missing at visit level (patient median in the ABT); coverage is recomputed in 1.4.9 -------
print("Remaining missing values after imputation:")
display(df[data_cols].isna().sum()[lambda s: s > 0].to_frame("n_missing"))

**Imputation decisions**

| Variable | Strategy | Why |
|---|---|---|
| Age, Education Level | Fill with the patient's own value from other visits | Static per patient and always available; no information is invented. |
| Insurance Provider | Infer from `coverage / price`, fall back to the patient's other visits | Each provider pays a fixed share (A 80%, B 90%, C 70%, D 100%, None 0%), so the ratio is a fingerprint of the provider. |
| Approximate Annual Income | Leave missing per visit; use the patient **median** in the ABT (2.1.1) | Income varies a little from visit to visit; a median is robust and every patient has at least one value. |
| Insurance Coverage | Recompute `rate x price` in 1.4.9 | Deterministic business rule; needs a validated price first. |

The ratio table above shows the evidence: for every provider the minimum equals the median rate (A 0.8, B 0.9, C 0.7, D 1.0, None 0); the few larger maxima are the 6 off-rule rows treated in 1.4.9.

Group-mean imputation was rejected: it would blur genuine differences between patients.

## <font color='turquoise'>1.4 Handle Inconsistent Data

[Back to TOC](#toc)

## <font color='white'>
This section follows the order in which problems can be fixed safely: first impossible values and data types, then the **static** patient attributes (age, income, gender, marital status, profession), then labels, ratings and cross-variable contradictions (including insurance coverage).

<a id="1_4_1"></a>

### 1.4.1 Identifying Inconsistent Data (Negative Values, Incorrect Dates)

In [ ]:
num_cols = ["Age", "Consultation Duration", "Satisfaction Level", "Approximate Annual Income",
            "Consultation Price", "Insurance Coverage"]
vd = df["Visit Date"]
audit = pd.DataFrame([
    ("Negative values in any numeric variable", int((df[num_cols] < 0).any(axis=1).sum())),
    ("Visit Date not a valid date (NaT)", int(vd.isna().sum())),
    ("Visit Date outside 1 Jan - 30 Jun 2024", int(((vd < "2024-01-01") | (vd > "2024-06-30")).sum())),
    ("Visit Date after the reference date (1 Jul 2024)", int((vd >= REF_DATE).sum())),
    ("Age impossible for a human (> 110)", int((df["Age"] > 110).sum())),
    ("Consultation Duration <= 0", int((df["Consultation Duration"] <= 0).sum())),
    ("Satisfaction outside the 1-5 scale", int((~df["Satisfaction Level"].between(1, 5)).sum())),
    ("Coverage greater than price (price <= 1000)",
     int(((df["Insurance Coverage"] > df["Consultation Price"]) & (df["Consultation Price"] <= 1000)).sum())),
], columns=["check", "rows_failing"])
display(audit)
print("Visit Date dtype:", vd.dtype, "| weekday coverage:", sorted(vd.dt.dayofweek.unique()))

**Findings.** There are **no negative values** and **no invalid or out-of-range dates**: every visit falls between 1 Jan and 30 Jun 2024, before the 1 Jul reference date. The real inconsistencies are impossible magnitudes: ages above 110 and a rating outside the 1-5 scale (handled in 1.4.3 and 1.4.7). Extreme prices, incomes and durations are treated as outliers in section 1.5, and the coverage rule in 1.4.9.

<a id="1_4_2"></a>

### 1.4.2 Correcting Data Types for Optimal Performance

In [ ]:
mem_before = df.memory_usage(deep=True).sum() / 1024

assert df["Age"].notna().all()                       # filled in 1.3.3, so it can be an integer now
df["Patient ID"] = df["Patient ID"].astype("int32")
df["Age"] = df["Age"].astype("int16")
df["Consultation Duration"] = df["Consultation Duration"].astype("int16")
df["Satisfaction Level"] = df["Satisfaction Level"].astype("int8")   # promoted to float in 1.4.7 (code 6 -> NaN)
df["Visit Date"] = pd.to_datetime(df["Visit Date"])

CAT_COLS = ["Gender", "City of Residence", "Profession", "Insurance Provider", "Family History",
            "Education Level", "Marital Status", "Department"]
for c in CAT_COLS:
    df[c] = df[c].astype("category")

mem_after = df.memory_usage(deep=True).sum() / 1024
print(f"Memory: {mem_before:,.0f} KB -> {mem_after:,.0f} KB")
display(df.dtypes.to_frame("dtype").T)

**Why these types.** `Age` became an integer (no gaps remain), `Patient ID` and the duration/rating columns are downcast to compact integers, `Visit Date` is a true datetime (needed for recency and monthly features), and the eight low-cardinality text columns are `category`, which stores each label once and speeds up grouping. The satisfaction column is temporarily `int8` and is promoted to a float in 1.4.7, once the invalid code becomes a missing value.

<a id="1_4_3"></a>

### 1.4.3 Checking Demographic and Socio-Economic Variables

In [ ]:
# Age: impossible values (> 110) are "hundreds-digit" typos (e.g. 138 vs 38 for the same patient)
bad_age = df["Age"] > 110
print("Rows with age > 110:", int(bad_age.sum()), "| patients:", sorted(df.loc[bad_age, "Patient ID"].unique().tolist()))
print("Range of impossible ages:", int(df.loc[bad_age, "Age"].min()), "-", int(df.loc[bad_age, "Age"].max()))
print("\nAges recorded for patient 1198 (mix of valid and impossible values proves a typo):")
print(df.loc[df["Patient ID"] == 1198, "Age"].value_counts().to_string())

df["flag_age_corrected"] = bad_age
df.loc[bad_age, "Age"] -= 100
print("\nCorrected ages:", sorted(df.loc[bad_age, "Age"].unique().tolist()))
assert df["Age"].between(0, 110).all()

In [ ]:
# Income: zero income - legitimate or an error?  (patient-level view)
pt = df.groupby("Patient ID", observed=True).agg(age=("Age", "median"), prof=("Profession", lambda s: s.mode().iloc[0]),
                                                 inc=("Approximate Annual Income", "median"))
pt["age_group"] = pd.cut(pt["age"], AGE_BINS, labels=AGE_LAB)
zero = pt["inc"] == 0
print(f"Patients with zero median income: {int(zero.sum())} of {len(pt)}")
display(pd.crosstab(pt["age_group"], zero.map({True: "income = 0", False: "income > 0"})))
print("Professions of zero-income patients:", {k: v for k, v in pt.loc[zero, "prof"].value_counts().items() if v})

**Decisions.**
- **Age:** all ages above 110 are clearly a typo (a stray hundreds digit). Subtracting 100 gives plausible ages that match each patient's profession and education, so the value is **corrected and flagged** (`flag_age_corrected`) instead of deleted.
- **Income:** a value of £0 is **legitimate**: it occurs for every child and for students, who have no income. Zero is kept as a true zero. Extreme positive incomes (scale errors) are handled in 1.5.

<a id="1_4_4"></a>

### 1.4.4 Identifying Inconsistent Patient Attributes (Static Features)

In [ ]:
# Attributes that describe the PATIENT should not change between visits.
static_cols = ["Age", "Gender", "City of Residence", "Profession", "Education Level",
               "Marital Status", "Family History", "Insurance Provider"]


def static_conflicts():
    """Number of patients with more than one value, for each static attribute."""
    return df.groupby("Patient ID", observed=True)[static_cols].nunique().gt(1).sum().to_frame("patients with >1 value")


display(static_conflicts())

for c in ["Gender", "Marital Status", "Profession"]:
    pats = df.groupby("Patient ID", observed=True)[c].nunique()
    pats = pats[pats > 1].index
    print(f"\n{c}: visits per label for the {len(pats)} conflicting patients")
    display(df[df["Patient ID"].isin(pats)].groupby(["Patient ID", c], observed=True).size().unstack(fill_value=0))

**Findings.** City, family history, education, insurance provider and (after 1.4.3) age are constant for every patient. **Gender, marital status and profession** change between visits for a handful of patients. In most cases the minority label appears in only 1-3 of ~20 visits, which looks like a data-entry error rather than a real change. Profession is special: two young children alternate between *Student* and *Retired*, which cannot both be true.

<a id="1_4_5"></a>

### 1.4.5 Correcting Inconsistencies Across Patient Visits

Rule: a static attribute takes the patient's **most frequent value**. This guarantees one value per patient and avoids conflicts when the visits are aggregated into the ABT.

In [ ]:
def patient_mode(col):
    """Most frequent value of a column for each patient."""
    return df.groupby("Patient ID", observed=True)[col].agg(lambda s: s.dropna().mode().iloc[0])


# Gender and marital status: most frequent value
for col in ["Gender", "Marital Status"]:
    df[col] = df["Patient ID"].map(patient_mode(col)).astype("category")

# Profession: most frequent value, but a child cannot be 'Retired' -> Student if age < 18 and 'Student' is among the labels
prof_mode = patient_mode("Profession").astype(str)
age_pat = df.groupby("Patient ID")["Age"].median()
prof = df["Profession"].astype(str)
conflict = prof.groupby(df["Patient ID"]).nunique() > 1
has_student = prof.eq("Student").groupby(df["Patient ID"]).any()
child_students = age_pat.index[conflict & has_student & (age_pat < 18)].tolist()
prof_mode[child_students] = "Student"
df["Profession"] = df["Patient ID"].map(prof_mode).astype("category")
print("Children set to Student:", child_students)

after = static_conflicts().rename(columns={"patients with >1 value": "patients with >1 value after cleaning"})
display(after)
assert (after.iloc[:, 0] == 0).all()

After the correction **every static attribute has exactly one value per patient**, so taking `first()` or `mode()` in the ABT is now safe.

<a id="1_4_6"></a>

### 1.4.6 Identifying Invalid Category Labels

In [ ]:
expected = {
    "Gender": {"Male", "Female", "Other"},
    "Insurance Provider": {"Provider A", "Provider B", "Provider C", "Provider D", "None"},
    "Marital Status": {"Single", "Married", "Divorced", "Widowed"},
    "Family History": {"None", "Diabetes", "Cancer", "Heart Disease", "Hypertension"},
    "Education Level": set(EDU),
}
rows = []
for c in CAT_COLS:
    lab = df[c].astype(str)
    stripped = lab.str.strip()
    rows.append(dict(variable=c, n_labels=lab.nunique(),
                     rows_with_extra_spaces=int((lab != stripped).sum()),
                     case_variants=int(lab.nunique() - stripped.str.lower().nunique()),
                     labels_outside_expected=sorted(set(lab) - expected[c]) if c in expected else "n/a (open list)"))
display(pd.DataFrame(rows))

for c in ["City of Residence", "Profession", "Department"]:
    print(f"{c}: {sorted(df[c].astype(str).unique())}")

**Result.** No label problems were found: no stray spaces, no upper/lower-case variants, no spelling variants and no label outside the documented domain (e.g. gender is Male/Female/Other, providers are A-D plus `None`). **No relabelling is needed.** The only thing to keep in mind is that `None` is a valid category and the `Other` gender share is large (about a third of patients), which is worth confirming with the data owner.

<a id="1_4_7"></a>

### 1.4.7 Correcting Satisfaction Level Values

In [ ]:
print("Raw rating counts:")
display(df["Satisfaction Level"].value_counts().sort_index().to_frame("visits").T)

bad_sat = df["Satisfaction Level"] == 6
print(f"Invalid rating (6): {int(bad_sat.sum())} visits = {bad_sat.mean():.1%}")
print("Where do they come from?  Visits per department with rating 6:")
display(df.loc[bad_sat, "Department"].value_counts().head(6).to_frame("rating_6_visits").T)

df["Satisfaction Level"] = df["Satisfaction Level"].astype(float)
df["flag_rating_invalid"] = bad_sat
df.loc[bad_sat, "Satisfaction Level"] = np.nan
print("\nValid ratings now:", int(df["Satisfaction Level"].notna().sum()), "| range:",
      df["Satisfaction Level"].min(), "-", df["Satisfaction Level"].max())

**Decision.** A rating of 6 is impossible on a 1-5 scale. It is set to **missing** (not capped to 5, not replaced by a mean) because the patient's real opinion is unknown, and 7% of visits is far too many to invent. Satisfaction features in the ABT are computed on valid ratings only, and `n_rated_visits` / `n_invalid_ratings` keep track of how much evidence each patient has. The invalid codes are concentrated in two departments, which suggests a faulty field in those departments' form rather than random noise.

<a id="1_4_8"></a>

### 1.4.8 Checking Logical Contradictions

In [ ]:
# (a) Profession vs age: children who are Doctor/Lawyer/Teacher/Retired, or a 95-year-old student
pa = df.groupby("Patient ID", observed=True).agg(age=("Age", "median"), prof=("Profession", lambda s: s.mode().iloc[0]))
odd = pa[((pa.age < 18) & (pa.prof != "Student")) | ((pa.age > 65) & (pa.prof == "Student"))]
display(odd.rename(columns={"age": "median_age", "prof": "profession"}))
df["flag_profession_age_odd"] = df["Patient ID"].isin(odd.index)

# (b) Education vs age: does the education level fit the patient's age?
pe = df.groupby("Patient ID", observed=True).agg(age=("Age", "median"), edu=("Education Level", lambda s: s.mode().iloc[0]))
pe["age_group"] = pd.cut(pe["age"], [-1, 5, 11, 17, 34, 200], labels=["0-5", "6-11", "12-17", "18-34", "35+"])
print("\nEducation level by age group (patients):")
display(pd.crosstab(pe["age_group"], pe["edu"])[[e for e in EDU if e in set(pe["edu"])]])

# (c) one insurance provider per patient (insurance is a patient-level attribute)
print("Patients with more than one insurance provider:", int((df.groupby("Patient ID", observed=True)["Insurance Provider"].nunique() > 1).sum()))
print("Visit rows already flagged as same patient/date/department:", int(df["flag_same_day_same_dept"].sum()))

**Findings.**
- **Profession vs age:** 4 patients (aged 1-11) are recorded as Lawyer, Doctor, Teacher or Retired. The label is the same on every visit, so it cannot be repaired from the data. Decision: **keep and flag** (`profession_age_odd`) and use `age_group` instead of `profession` for segmentation. Removing the patients is not justified because their visits are otherwise valid.
- **Education vs age:** education stages follow age (kindergarten/pre-school for the youngest, elementary and middle school for 6-11, middle/high school for teenagers, university from 18), so no contradiction.
- **Insurance:** every patient has exactly one provider.
- **Same-day visits** were already flagged in 1.2.2.

<a id="1_4_9"></a>

### 1.4.9 Checking Invalid Insurance Coverages

In [ ]:
def rate_of(provider):
    return provider.astype(str).map(RATE).astype(float)


price, cov = df["Consultation Price"], df["Insurance Coverage"]
exp_cov = price * rate_of(df["Insurance Provider"])
valid_price = price <= 1000                      # prices above this are scale errors (see 1.5)

# (a) the rows with a huge price: the PRICE is wrong, not the coverage
contra = df.loc[~valid_price, ["Patient ID", "Department", "Insurance Provider", "Consultation Price", "Insurance Coverage"]].copy()
contra["coverage_over_price"] = (contra["Insurance Coverage"] / contra["Consultation Price"]).round(4)
contra["expected_rate"] = rate_of(contra["Insurance Provider"])
print("Rows whose price is >1000. For the insured rows, coverage/price is ~100x below the provider rate: the PRICE is wrong, not the coverage")
display(contra)

# (b) rule check on all rows with a plausible price
miss_cov = cov.isna()
inconsistent = valid_price & ~miss_cov & ((cov - exp_cov).abs() > 0.01 * price + 0.01)
print(f"\nRows obeying 'coverage = rate x price': {int((valid_price & ~miss_cov & ~inconsistent).sum())}")
print(f"Missing coverage: {int(miss_cov.sum())} | coverage off the provider rate: {int(inconsistent.sum())}"
      f" (of which > price: {int((cov > price)[inconsistent].sum())})")
display(df.loc[inconsistent, ["Patient ID", "Insurance Provider", "Consultation Price", "Insurance Coverage"]]
        .assign(expected=exp_cov[inconsistent].round(2)).head(8))

# (c) repair: recompute coverage = provider rate x price where it is missing or off the rule
fix = (miss_cov | inconsistent) & valid_price
df["flag_coverage_recomputed"] = fix
df.loc[fix, "Insurance Coverage"] = exp_cov[fix]

**Decision.** Insurance coverage follows an exact rule (`coverage = provider rate x price`) on every valid row, so missing or off-rule values are **recomputed** rather than imputed with an average, and flagged in `flag_coverage_recomputed`. The seven rows with a price above £1,000 are *not* edited here: for the two insured rows the coverage is exactly what a price 100x smaller would give, which is the evidence that the **price** carries a x100 error (fixed in 1.5.3, where the rule is re-checked on every row).

<a id="1_5"></a>

## 1.5 Identifying and Handling Outliers

[Back to TOC](#toc)

<a id="1_5_1"></a>

### 1.5.1 Detecting Outliers in Numerical Variables

Two complementary tests: the statistical **IQR rule** (1.5 x IQR) and **domain rules** (what is physically/commercially possible).

In [ ]:
num = ["Age", "Consultation Duration", "Approximate Annual Income", "Consultation Price", "Insurance Coverage"]
rows = []
for c in num:
    s = df[c].dropna().astype(float)
    q1, q3 = s.quantile([.25, .75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    rows.append(dict(variable=c, min=s.min(), median=s.median(), p99_9=s.quantile(.999), max=s.max(),
                     iqr_low_fence=round(lo, 1), iqr_high_fence=round(hi, 1),
                     n_outside_iqr_fences=int(((s < lo) | (s > hi)).sum())))
display(pd.DataFrame(rows).round(1))

# domain evidence ----------------------------------------------------------------------------------------
print("Consultation Duration: the 12 largest distinct values (note the jump after 120):")
print([int(v) for v in sorted(df["Consultation Duration"].unique())[-12:]])

band = df[df["Consultation Price"] <= 1000].groupby("Department", observed=True)["Consultation Price"].agg(["min", "max"]).round(0)
print("\nNormal price band per department (price <= 1000):")
display(band.T)

pb = df[df["Consultation Price"] > 1000][["Patient ID", "Department", "Consultation Price"]].copy()
pb["price_div_100"] = (pb["Consultation Price"] / 100).round(1)
pb["dept_band_max"] = pb["Department"].astype(str).map(band["max"].to_dict())
print("\nPrices > 1000 (candidate x100 scale errors):")
display(pb)

inc = df["Approximate Annual Income"]
others = df[inc < 1_000_000].groupby("Patient ID")["Approximate Annual Income"].agg(other_visits_min="min", other_visits_max="max")
ib = df.loc[inc >= 1_000_000, ["Patient ID", "Approximate Annual Income"]].rename(columns={"Approximate Annual Income": "income"}).copy()
ib["divide_by_10"] = (ib["income"] / 10).round(0)
ib["divide_by_100"] = (ib["income"] / 100).round(0)
ib = ib.join(others.round(0), on="Patient ID")
print(f"Incomes >= 1,000,000: {len(ib)} rows from {ib['Patient ID'].nunique()} patients. "
      "Candidate corrections vs the range of the same patient's other visits:")
display(ib.sort_values(["Patient ID", "income"]).reset_index(drop=True))
print("Incomes in the rest of the data: max =", f"{int(inc[inc < 1_000_000].max()):,}")

**Reading the tests.** The IQR rule is only reliable for **duration** (it flags exactly the 10 extreme rows). For **price** it also flags 240 normal Emergency/Cardiology visits, because prices legitimately differ by department (the pooled quartiles are not appropriate), and for **income** it flags 2,285 rows because zero income for children and students makes the distribution highly skewed. For these two variables the domain evidence decides: every normal price lies inside its department's band (£50-£400) and no income outside the 7 + 21 extreme rows exceeds about £113k. A price above £1,000 or an income of £1M or more is therefore an error, not a rare patient.

For income, dividing by 10 (1.0-9.9M) or by 100 (>= 10M) gives £113k-£120k. This lands at the top of the same patient's own range (other visits about £50k-£113k) while the other divisor would give about £11-12k, far below it, so it identifies the intended value.

<a id="1_5_2"></a>

### 1.5.2 Visualizing Potential Outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
specs = [("Age", None, "Age (years)", "Age"),
         ("Consultation Duration", None, "Minutes", "Consultation duration"),
         ("Consultation Price", "log", "GBP (log scale)", "Consultation price"),
         ("Approximate Annual Income", "symlog", "GBP (symlog scale)", "Annual income")]
for ax, (c, scale, ylab, title) in zip(axes, specs):
    sns.boxplot(y=df[c].dropna().astype(float), ax=ax, color="#bcd3ee", flierprops=dict(marker="o", markersize=3, alpha=.5))
    if scale == "log":
        ax.set_yscale("log")
    elif scale == "symlog":
        ax.set_yscale("symlog", linthresh=1000)
    ax.set_title(title); ax.set_ylabel(ylab); ax.set_xlabel("Variable (raw data after 1.3-1.4)")
fig.suptitle("Potential outliers in the numerical variables", fontweight="bold")
plt.tight_layout(); plt.show()

**Interpretation.** Each variable has a compact bulk of plausible values and a few points that are orders of magnitude away: prices of £25,000-£40,000 (normal range £50-£400), incomes of £1M-£12M (normal range £0-£115k) and durations of 228-600 minutes (all other visits last 15-120 minutes). Ages are already clean after 1.4.3. The extremes are not gradual tails but **isolated, far-away values**, a typical sign of entry or scale errors rather than genuine patients.

<a id="1_5_3"></a>

### 1.5.3 Deciding How to Handle Outliers

In [ ]:
# --- Price: x100 scale errors -> divide by 100 ----------------------------------------------------------------
bad_price = df["Consultation Price"] > 1000
df["flag_price_corrected"] = bad_price
df.loc[bad_price, "Consultation Price"] /= 100

# --- Income: x10 / x100 scale errors -> divide back; zeros are legitimate (children/students) -----------------
inc = df["Approximate Annual Income"]
df["flag_income_corrected"] = inc >= 1_000_000
df["Approximate Annual Income"] = np.where(inc >= 10_000_000, inc / 100, np.where(inc >= 1_000_000, inc / 10, inc))

# --- Duration: max plausible 120 min -> department median -----------------------------------------------------
bad_dur = df["Consultation Duration"] > 120
df["flag_duration_imputed"] = bad_dur
med_dur = df.loc[~bad_dur].groupby("Department", observed=True)["Consultation Duration"].median().round()
df.loc[bad_dur, "Consultation Duration"] = df.loc[bad_dur, "Department"].astype(str).map(med_dur.to_dict()).astype("int16")

print(f"Corrected prices: {int(bad_price.sum())} | corrected incomes: {int(df['flag_income_corrected'].sum())} | imputed durations: {int(bad_dur.sum())}")

**Decisions: correct when the error has a recoverable pattern, impute when it does not, never delete.**

| Variable | Outlier | Decision | Reason |
|---|---|---|---|
| Consultation Price | 7 rows of £25k-£40k | Divide by 100 and flag | Result lands exactly inside the department's normal price band, and for the two insured rows the (correct) coverage already implied a normal price. |
| Annual Income | 21 rows of £1M-£12M | Divide by 10 (1.0-9.9M) or 100 (>= 10M) and flag | Gives £113k-£120k, at the top of the same patient's own range (other visits about £50k-£113k); the other divisor would give about £11-12k, far outside it. |
| Consultation Duration | 10 rows of 228-600 min | Impute department median and flag | All other visits are 15-120 min; the extremes are not a constant multiple of a plausible value, so the true duration is unknown. |
| Age | already corrected in 1.4.3 | - | - |
| Zero income | 110 patients | Keep | Children and students legitimately earn nothing. |

In [ ]:
# --- Final validation: rules re-checked on every row after all corrections ------------------------------------
exp_cov = df["Consultation Price"] * rate_of(df["Insurance Provider"])
assert ((df["Insurance Coverage"] - exp_cov).abs() <= 0.01 * df["Consultation Price"] + 0.01).all()   # coverage = rate x price
assert df["Age"].between(0, 110).all()
assert df["Consultation Price"].between(50, 400).all()
assert df["Consultation Duration"].between(15, 120).all()
assert (df["Insurance Coverage"] <= df["Consultation Price"] + 1e-6).all()
assert df["Visit Date"].between("2024-01-01", "2024-06-30").all()
assert df["Approximate Annual Income"].dropna().between(0, 250_000).all()

# derived visit-level columns used by the ABT and the charts
for c in CAT_COLS:
    df[c] = df[c].astype("category")                   # dtypes may have been reset by the corrections above
df["out_of_pocket"] = df["Consultation Price"] - df["Insurance Coverage"]
df["Month"] = df["Visit Date"].dt.to_period("M").dt.to_timestamp()
df["age_group"] = pd.cut(df["Age"], AGE_BINS, labels=AGE_LAB)

df.to_csv("visit_level_clean.csv", index=False)
print(f"All validation rules pass. Clean visits: {len(df):,} | patients: {df['Patient ID'].nunique()}")

**Summary of all cleaning decisions**

| Issue | Decision |
|---|---|
| Exact duplicate visits | Remove (keep first) |
| Same patient, date and department with different values | Keep and flag |
| Age, education, insurance provider missing | Fill from the patient's other visits (provider: from coverage / price first) |
| Income missing | Keep missing; patient median in the ABT |
| Impossible ages (> 110) | Subtract 100 and flag |
| Gender, marital status, profession change between visits | Patient's most frequent value (child: Student) |
| Rating of 6 (scale is 1-5) | Set to missing, flag |
| Profession implausible for age | Keep and flag |
| Coverage missing or off the provider rate | Recompute rate x price, flag |
| Price x100, income x10 / x100 | Divide back and flag |
| Duration above 120 min | Department median, flag |

Number of rows touched by each flag:

In [ ]:
display(df.filter(like="flag_").sum().to_frame("rows_flagged").T)

<a id="2"></a>

# 2. Part B - Feature Engineering & ABT Construction

[Back to TOC](#toc)

One row per `Patient ID`. Features are grouped as **static**, **visit behaviour**, **financial**, **satisfaction**, **recency & activity** (reference date 1 July 2024) and **data-quality flags**. Because every static attribute now has a single value per patient (1.4.5), aggregating with `first()` is safe.

<a id="2_1"></a>

## 2.1 Patient-Level ABT Construction

[Back to TOC](#toc)

In [ ]:
g = df.groupby("Patient ID", observed=True)
dept_s = df["Department"].astype(str)                  # plain strings for the department logic
abt = pd.DataFrame(index=g.size().index)
abt.index.name = "patient_id"

<a id="2_1_1"></a>

### 2.1.1 Static Attributes

In [ ]:
abt["age"] = g["Age"].first().astype(int)
abt["age_group"] = pd.cut(abt["age"], AGE_BINS, labels=AGE_LAB).astype(str)
abt["gender"] = g["Gender"].first().astype(str)
abt["city"] = g["City of Residence"].first().astype(str)
abt["profession"] = g["Profession"].first().astype(str)
abt["education_level"] = g["Education Level"].first().astype(str)
abt["education_ordinal"] = abt["education_level"].map({k: i for i, k in enumerate(EDU)})
abt["marital_status"] = g["Marital Status"].first().astype(str)
abt["family_history"] = g["Family History"].first().astype(str)
abt["has_family_history"] = (abt["family_history"] != "None").astype(int)
abt["insurance_provider"] = g["Insurance Provider"].agg(lambda s: s.mode().iloc[0]).astype(str)
abt["has_insurance"] = (abt["insurance_provider"] != "None").astype(int)
abt["insurance_coverage_rate"] = abt["insurance_provider"].map(RATE)
abt["annual_income"] = g["Approximate Annual Income"].median().round(0)
abt["income_std"] = g["Approximate Annual Income"].std().round(0)
assert abt["annual_income"].notna().all()
display(abt.head(3))

<a id="2_1_2"></a>

### 2.1.2 Visit Behaviour Features

In [ ]:
gd = dept_s.groupby(df["Patient ID"])
abt["n_visits"] = g.size()
abt["n_distinct_departments"] = gd.nunique()
abt["top_department"] = gd.agg(lambda s: s.value_counts(sort=False).idxmax())   # ties: first department seen
abt["top_department_share"] = (gd.agg(lambda s: s.value_counts(sort=False).max()) / abt["n_visits"]).round(4)
abt["avg_duration_min"] = g["Consultation Duration"].mean().round(2)
abt["std_duration_min"] = g["Consultation Duration"].std().round(2)
abt["total_duration_min"] = g["Consultation Duration"].sum()
abt["n_emergency_visits"] = gd.agg(lambda s: (s == "Emergency").sum())
abt["share_emergency"] = (abt["n_emergency_visits"] / abt["n_visits"]).round(4)
dept_counts = pd.crosstab(df["Patient ID"], dept_s)
dept_counts.columns = ["visits_" + c.lower().replace(" ", "_") for c in dept_counts.columns]
abt = abt.join(dept_counts)
display(abt[["n_visits", "n_distinct_departments", "top_department", "top_department_share", "avg_duration_min"]].head(3))

<a id="2_1_3"></a>

### 2.1.3 Financial Features

In [ ]:
abt["total_price"] = g["Consultation Price"].sum().round(2)
abt["avg_price"] = g["Consultation Price"].mean().round(2)
abt["total_insurance_coverage"] = g["Insurance Coverage"].sum().round(2)
abt["avg_insurance_coverage"] = g["Insurance Coverage"].mean().round(2)
abt["total_out_of_pocket"] = g["out_of_pocket"].sum().round(2)       # out-of-pocket = price - coverage
abt["avg_out_of_pocket"] = g["out_of_pocket"].mean().round(2)
abt["coverage_share"] = (abt["total_insurance_coverage"] / abt["total_price"]).round(4)
abt["oop_to_income_pct"] = np.where(abt["annual_income"] > 0,
                                    (abt["total_out_of_pocket"] / abt["annual_income"].replace(0, np.nan) * 100).round(3), np.nan)
display(abt[["total_price", "avg_price", "total_insurance_coverage", "total_out_of_pocket", "coverage_share"]].head(3))

<a id="2_1_4"></a>

### 2.1.4 Satisfaction Features

In [ ]:
s = g["Satisfaction Level"]                                    # valid ratings only (code 6 is NaN)
abt["avg_satisfaction"] = s.mean().round(3)
abt["min_satisfaction"] = s.min()
abt["max_satisfaction"] = s.max()
abt["std_satisfaction"] = s.std().round(3)
abt["n_rated_visits"] = s.count()
abt["share_low_ratings"] = (s.agg(lambda x: (x <= 2).sum()) / abt["n_rated_visits"]).round(4)
abt["share_high_ratings"] = (s.agg(lambda x: (x >= 4).sum()) / abt["n_rated_visits"]).round(4)
abt["n_invalid_ratings"] = g["flag_rating_invalid"].sum()
display(abt[["avg_satisfaction", "min_satisfaction", "share_low_ratings", "n_rated_visits", "n_invalid_ratings"]].head(3))

<a id="2_1_5"></a>

### 2.1.5 Recency and Activity Features

In [ ]:
last = g["Visit Date"].max()
first = g["Visit Date"].min()
abt["days_since_last_visit"] = (REF_DATE - last).dt.days
abt["days_since_first_visit"] = (REF_DATE - first).dt.days
abt["observation_months"] = (abt["days_since_first_visit"] / 30.4375).round(3)
abt["visits_per_month"] = (abt["n_visits"] / abt["observation_months"]).round(3)
abt["avg_days_between_visits"] = ((last - first).dt.days / (abt["n_visits"] - 1)).round(2)
q1 = df[df["Visit Date"] < "2024-04-01"].groupby("Patient ID").size()
abt["visits_q1"] = q1.reindex(abt.index).fillna(0).astype(int)
abt["visits_q2"] = abt["n_visits"] - abt["visits_q1"]
abt["visit_trend_q2_minus_q1"] = abt["visits_q2"] - abt["visits_q1"]
display(abt[["days_since_last_visit", "days_since_first_visit", "visits_per_month", "visits_q1", "visits_q2"]].head(3))

**Data-quality flags (patient level).** Besides the business features, the ABT keeps a few flags so the next team can exclude or investigate patients whose data was repaired.

In [ ]:
for c, n in [("flag_age_corrected", "dq_age_corrected"), ("flag_price_corrected", "dq_price_corrected"),
             ("flag_income_corrected", "dq_income_corrected"), ("flag_profession_age_odd", "profession_age_odd")]:
    abt[n] = g[c].max().astype(int)
flag_cols = ["flag_duration_imputed", "flag_coverage_recomputed", "flag_rating_invalid",
             "flag_price_corrected", "flag_income_corrected"]
abt["dq_n_flagged_visits"] = df.assign(_f=df[flag_cols].any(axis=1)).groupby("Patient ID")["_f"].sum()

abt = abt.reset_index()
assert abt["patient_id"].is_unique and len(abt) == 500
print("ABT shape:", abt.shape, "| one row per patient:", abt["patient_id"].is_unique)
print("Columns with missing values:", abt.isna().sum()[lambda s: s > 0].to_dict())

The only remaining missing values are structural: `oop_to_income_pct` is undefined when income is zero (children/students) and `std_*` features are undefined for patients with a single valid observation. They are left as NaN on purpose, because a fake value would mislead a model.

<a id="2_2"></a>

## 2.2 Data Dictionary

[Back to TOC](#toc)

In [ ]:
D = []
def d(name, group, definition, calc):
    D.append(dict(feature=name, group=group, definition=definition, calculation=calc))

d("patient_id", "Key", "Unique patient identifier", "Patient ID")
d("age", "Static", "Age in years", "Patient's age (typos >110 corrected by -100; gaps filled from other visits)")
d("age_group", "Static", "Age band", "Bins 0-17, 18-34, 35-49, 50-64, 65-79, 80+")
d("gender", "Static", "Gender", "Most frequent label across the patient's visits")
d("city", "Static", "City of residence", "Constant per patient")
d("profession", "Static", "Profession", "Most frequent label (Student if age<18 and conflict)")
d("education_level", "Static", "Highest education", "Constant per patient; gaps filled from other visits")
d("education_ordinal", "Static", "Ordinal education 0-7", "Kindergarten=0 ... PhD=7")
d("marital_status", "Static", "Marital status", "Most frequent label across visits")
d("family_history", "Static", "Family disease history (None/Diabetes/Cancer/Heart Disease/Hypertension)", "Constant per patient")
d("has_family_history", "Static", "1 if any family disease", "family_history != 'None'")
d("insurance_provider", "Static", "Insurance provider (None, A-D)", "Mode over visits; missing inferred from coverage/price ratio")
d("has_insurance", "Static", "1 if insured", "insurance_provider != 'None'")
d("insurance_coverage_rate", "Static", "Share of price paid by insurer", "A=0.8, B=0.9, C=0.7, D=1.0, None=0 (observed rule)")
d("annual_income", "Static", "Approximate annual income (GBP)", "Median of the patient's visit-level incomes after scale correction")
d("income_std", "Static", "Variability of recorded income", "Std of visit-level income")
d("n_visits", "Behaviour", "Number of consultations Jan-Jun 2024", "Row count per patient (after de-duplication)")
d("n_distinct_departments", "Behaviour", "Different departments visited", "nunique(Department)")
d("top_department", "Behaviour", "Most frequent department", "Mode of Department (first in count order on ties)")
d("top_department_share", "Behaviour", "Share of visits in top department", "visits in top dept / n_visits")
d("avg_duration_min", "Behaviour", "Average consultation length (min)", "mean(Consultation Duration)")
d("std_duration_min", "Behaviour", "Std of consultation length", "std(Consultation Duration)")
d("total_duration_min", "Behaviour", "Total minutes with the hospital", "sum(Consultation Duration)")
d("n_emergency_visits", "Behaviour", "Emergency visits", "count(Department=='Emergency')")
d("share_emergency", "Behaviour", "Share of visits in Emergency", "n_emergency_visits / n_visits")
d("visits_<department> (13 cols)", "Behaviour", "Visits per department", "Crosstab patient x department")
d("total_price", "Financial", "Total price of all consultations (GBP)", "sum(Consultation Price)")
d("avg_price", "Financial", "Average price per consultation", "mean(Consultation Price)")
d("total_insurance_coverage", "Financial", "Total paid by insurer", "sum(Insurance Coverage)")
d("avg_insurance_coverage", "Financial", "Average coverage per visit", "mean(Insurance Coverage)")
d("total_out_of_pocket", "Financial", "Total paid by patient", "sum(price - coverage)")
d("avg_out_of_pocket", "Financial", "Average out-of-pocket per visit", "mean(price - coverage)")
d("coverage_share", "Financial", "Share of cost covered by insurance", "total_insurance_coverage / total_price")
d("oop_to_income_pct", "Financial", "Out-of-pocket as % of annual income", "total_out_of_pocket / annual_income *100 (NaN if income = 0)")
d("avg_satisfaction", "Satisfaction", "Mean rating (1-5)", "mean of valid ratings (code 6 excluded)")
d("min_satisfaction", "Satisfaction", "Lowest rating", "min of valid ratings")
d("max_satisfaction", "Satisfaction", "Highest rating", "max of valid ratings")
d("std_satisfaction", "Satisfaction", "Rating variability", "std of valid ratings")
d("n_rated_visits", "Satisfaction", "Visits with a valid rating", "count of valid ratings")
d("share_low_ratings", "Satisfaction", "Share of low ratings (<=2)", "visits rated 1-2 / n_rated_visits")
d("share_high_ratings", "Satisfaction", "Share of high ratings (>=4)", "visits rated 4-5 / n_rated_visits")
d("n_invalid_ratings", "Satisfaction", "Visits with invalid rating (6)", "count of removed ratings")
d("days_since_last_visit", "Recency", "Recency (days)", "2024-07-01 minus last visit date")
d("days_since_first_visit", "Recency", "Tenure in window (days)", "2024-07-01 minus first visit date")
d("observation_months", "Recency", "Months from first visit to reference date", "days_since_first_visit / 30.4375")
d("visits_per_month", "Recency", "Visit rate", "n_visits / observation_months")
d("avg_days_between_visits", "Recency", "Mean gap between visits", "(last - first) days / (n_visits - 1)")
d("visits_q1 / visits_q2", "Recency", "Visits Jan-Mar / Apr-Jun", "count by quarter")
d("visit_trend_q2_minus_q1", "Recency", "Change in activity", "visits_q2 - visits_q1")
d("dq_age_corrected / dq_price_corrected / dq_income_corrected", "Data quality",
  "1 if any value of the patient was corrected", "max of visit-level flags")
d("profession_age_odd", "Data quality", "1 if profession implausible for age", "child non-student or senior student")
d("dq_n_flagged_visits", "Data quality", "Visits with an imputed/corrected/invalid value", "count of flagged visits")

data_dictionary = pd.DataFrame(D)

# completeness check: every ABT column must be documented
documented = {p.split(" (")[0] for f in data_dictionary["feature"] for p in f.split(" / ")}
dept_cols = [c for c in abt.columns if c.startswith("visits_") and c not in ("visits_q1", "visits_q2", "visits_per_month")]
undocumented = [c for c in abt.columns if c not in documented and c not in dept_cols]
print(f"ABT columns: {abt.shape[1]} | department columns covered by one pattern row: {len(dept_cols)} | undocumented: {undocumented or 'none'}")
data_dictionary

**Most important features for the next team.** `n_visits` / `visits_per_month` (utilisation), `top_department` and `n_distinct_departments` (service mix), `total_out_of_pocket` and `coverage_share` (financial burden), `avg_satisfaction` / `share_low_ratings` (experience), `age_group`, `annual_income` and `has_insurance` (segmentation) and the `dq_*` flags (to test the sensitivity of any result to repaired data).

<a id="2_3"></a>

## 2.3 Exporting the ABT

[Back to TOC](#toc)

In [ ]:
abt.to_csv("ABT_city_hospital.csv", index=False)
data_dictionary.to_csv("data_dictionary.csv", index=False)

check = pd.read_csv("ABT_city_hospital.csv")
assert check["patient_id"].is_unique and len(check) == 500
print(f"Saved ABT_city_hospital.csv: {check.shape[0]} rows (one per patient) x {check.shape[1]} columns")
print("Also saved: data_dictionary.csv, visit_level_clean.csv")
display(check.head())

<a id="3"></a>

# 3. Part C - Exploratory Visualization & Insights

[Back to TOC](#toc)

Each chart has a title, labelled axes and a short interpretation below it. Significance tests are permutation tests written with numpy (5,000 shuffles, fixed seed), so no extra library is needed.

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "axes.axisbelow": True,
                     "axes.titleweight": "bold", "axes.titlesize": 12})
BLUE, ORANGE, GREY, TEAL, RED = "#2a6fbb", "#e08a1e", "#8a8f98", "#1b9e8f", "#c0392b"
CAT = [BLUE, ORANGE, TEAL, "#8e5ea2", "#d1495b"]


def show():
    plt.tight_layout()
    plt.show()
    plt.close()


# Significance tests with numpy only (no scipy): permutation tests, reproducible through the fixed seed
rng = np.random.default_rng(42)


def perm_chi2_p(a, b, n_perm=5000):
    """p-value of the chi-square test of independence between two categorical series (permutation)."""
    ia, ka = pd.factorize(a)
    ib, kb = pd.factorize(b)

    def chi2(ib_):
        obs = np.bincount(ia * len(kb) + ib_, minlength=len(ka) * len(kb)).reshape(len(ka), len(kb)).astype(float)
        exp = obs.sum(1, keepdims=True) * obs.sum(0, keepdims=True) / obs.sum()
        return ((obs - exp) ** 2 / exp).sum()

    obs_stat = chi2(ib)
    count = sum(chi2(rng.permutation(ib)) >= obs_stat for _ in range(n_perm))
    return float((count + 1) / (n_perm + 1))


def perm_anova_p(groups, y, n_perm=5000):
    """p-value of the one-way ANOVA F test (do the group means differ?), by permutation."""
    ig, kg = pd.factorize(groups)
    y = np.asarray(y, float)
    n, k = len(y), len(kg)
    ng = np.bincount(ig)

    def F(y_):
        means = np.bincount(ig, weights=y_) / ng
        ssb = (ng * (means - y_.mean()) ** 2).sum()
        sst = ((y_ - y_.mean()) ** 2).sum()
        return (ssb / (k - 1)) / ((sst - ssb) / (n - k))

    obs = F(y)
    count = sum(F(rng.permutation(y)) >= obs for _ in range(n_perm))
    return float((count + 1) / (n_perm + 1))

<a id="3_1"></a>

## 3.1 Patient Demographics

[Back to TOC](#toc)

<a id="3_1_1"></a>

### 3.1.1 Age Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(abt["age"], bins=range(0, 100, 5), color=BLUE, edgecolor="white")
ax.set_title("Age distribution of patients (n=500)"); ax.set_xlabel("Age (years)"); ax.set_ylabel("Number of patients")
show()

**Figure 1 - Age distribution.** Ages are spread almost evenly from 0 to 95 (median 50, mean 49.2), with no dominant age. Children (0-17) are 101 of 500 patients and 65+ patients 174: the hospital serves all life stages, so age should be used as a segmentation variable rather than dropped.

<a id="3_1_2"></a>

### 3.1.2 Gender Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
gc = abt["gender"].value_counts()
ax.bar(gc.index, gc.values, color=BLUE)
for i, v in enumerate(gc.values):
    ax.text(i, v + 3, f"{v} ({v/5:.0f}%)", ha="center")
ax.set_title("Patients by gender"); ax.set_xlabel("Gender"); ax.set_ylabel("Number of patients"); ax.set_ylim(0, gc.max() * 1.15)
show()

**Figure 2 - Gender.** The three gender categories are almost balanced (Female 169, Male 160, Other 171 patients). The large 'Other' share (~1/3) is unusual for real hospital data and should be confirmed with the data owner, but it is internally consistent after cleaning.

<a id="3_1_3"></a>

### 3.1.3 Income Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(abt["annual_income"], bins=30, color=BLUE, edgecolor="white")
axes[0].set_title("Annual income - all patients"); axes[0].set_xlabel("Annual income (GBP)"); axes[0].set_ylabel("Number of patients")
order = abt.groupby("profession")["annual_income"].median().sort_values().index
sns.boxplot(data=abt, x="annual_income", y="profession", order=order, color="#bcd3ee", medianprops=dict(color=RED), ax=axes[1])
axes[1].set_title("Annual income by profession"); axes[1].set_xlabel("Annual income (GBP)"); axes[1].set_ylabel("Profession")
show()

**Figure 3 - Income distribution.** Median patient income is £43,458; 110 patients (22%) have zero income (children and students). Among adults, income is nearly identical across professions (~£43-46k), so profession carries little information about income, while student/non-student carries a lot.

<a id="3_2"></a>

## 3.2 Department and Service Usage

[Back to TOC](#toc)

<a id="3_2_1"></a>

### 3.2.1 Consultations by Department

In [ ]:
dv = df["Department"].astype(str).value_counts()
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(dv.index[::-1], dv.values[::-1], color=BLUE)
for i, v in enumerate(dv.values[::-1]):
    ax.text(v + 10, i, str(v), va="center", fontsize=9)
ax.set_title("Consultations per department, Jan-Jun 2024"); ax.set_xlabel("Number of visits"); ax.set_ylabel("Department")
ax.set_xlim(0, dv.max() * 1.1)
show()

**Figure 4 - Visits per department.** Psychiatry (1,359) and General Practice (1,331) are the busiest departments and together take 27% of all consultations - about twice the volume of any other department. Orthopedics is the quietest (593).

<a id="3_2_2"></a>

### 3.2.2 Consultations Over Time

In [ ]:
mt = df.groupby("Month").size()
top4 = dv.index[:4]
pm = df.groupby(["Month", "Department"], observed=True).size().unstack()
pm.columns = pm.columns.astype(str)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mt.index, mt.values, marker="o", color=BLUE, lw=2)
axes[0].set_ylim(0, mt.max() * 1.2)
axes[0].set_title("Total consultations per month"); axes[0].set_xlabel("Month (2024)"); axes[0].set_ylabel("Number of visits")
for c, dep in zip(CAT, top4):
    axes[1].plot(pm.index, pm[dep], marker="o", lw=2, color=c, label=dep)
axes[1].set_title("Monthly visits, four busiest departments"); axes[1].set_xlabel("Month (2024)"); axes[1].set_ylabel("Number of visits")
axes[1].legend(frameon=False, fontsize=8)
for a in axes:
    a.set_xticks(mt.index); a.set_xticklabels([m.strftime("%b") for m in mt.index])
show()

**Figure 5 - Demand over time.** Total demand is flat: 1,728 visits in January, 1,616 in June, with Feb the lowest simply because it is the shortest month (daily rate is ~53-57 visits in every month). Individual departments fluctuate (e.g. Psychiatry falls from 254 in Jan to 197 in Jun) but there is no consistent trend.

In [ ]:
# heat map department x month (colour = share of the department's visits, cell = visits)
hm = pm.T
hm.columns = [m.strftime("%b") for m in hm.columns]
hm_sh = hm.div(hm.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(hm_sh, annot=hm, fmt="d", cmap="Blues", vmin=10, vmax=24, cbar_kws={"label": "% of department's visits"}, ax=ax)
ax.grid(False); ax.set_title("Visits by department and month (cell = visits)"); ax.set_xlabel("Month (2024)"); ax.set_ylabel("Department")
show()

**Figure 6 - Department × month.** The heat map shows no systematic seasonality by department: counts per cell stay within ~75-255. The visible peaks (Psychiatry in Jan, General Practice in Mar/May) are modest and should be re-checked with a longer time series.

<a id="3_2_3"></a>

### 3.2.3 Department Usage by Gender or Age Group

In [ ]:
# department by gender (share of the department's visits)
dg = pd.crosstab(df["Department"].astype(str), df["Gender"].astype(str), normalize="index") * 100
dg = dg.loc[dv.index]
fig, ax = plt.subplots(figsize=(9, 4.6))
bottom = np.zeros(len(dg))
x = np.arange(len(dg))
for c, gname in zip([BLUE, ORANGE, TEAL], ["Female", "Male", "Other"]):
    ax.bar(x, dg[gname].values, bottom=bottom, color=c, label=gname, edgecolor="white")
    bottom += dg[gname].values
ax.set_xticks(x); ax.set_xticklabels(dg.index, rotation=40, ha="right")
ax.set_title("Gender mix of each department's visits"); ax.set_xlabel("Department"); ax.set_ylabel("% of visits")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(.5, -.4))
show()
p_dept_gender = perm_chi2_p(df["Department"].astype(str), df["Gender"].astype(str))
print(f"Permutation chi-square p-value, department vs gender: {p_dept_gender:.2f}")

**Figure 7 - Department by gender.** Every department has a roughly one-third mix of each gender (Female share 29-36%). The differences are small and not statistically significant (permutation chi-square test, p = 0.23), so gender does not discriminate department use in this data.

In [ ]:
# department by age group (share of the age group's visits)
da = pd.crosstab(df["age_group"].astype(str), df["Department"].astype(str), normalize="index") * 100
da = da.loc[AGE_LAB, dv.index]
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(da, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of visits"}, ax=ax)
ax.grid(False); ax.set_title("Department mix by age group (% of the group's visits)"); ax.set_xlabel("Department"); ax.set_ylabel("Age group (years)")
plt.xticks(rotation=40, ha="right")
show()
p_dept_age = perm_chi2_p(df["age_group"].astype(str), df["Department"].astype(str))
print(f"Permutation chi-square p-value, department vs age group: {p_dept_age:.2f}")

**Figure 8 - Department by age group.** Department mix is similar across age groups (permutation chi-square test, p = 0.20). Descriptive hints, to be treated as hypotheses only: Orthopedics is lowest for 50-64 (4.7% vs ~6.5% overall), Emergency is lowest for 80+ (5.3%), Pulmonology is highest for 80+ (7.7%).

<a id="3_3"></a>

## 3.3 Patient Satisfaction

[Back to TOC](#toc)

<a id="3_3_1"></a>

### 3.3.1 Satisfaction by Department

In [ ]:
sat = df.dropna(subset=["Satisfaction Level"])
sd = sat.groupby("Department", observed=True)["Satisfaction Level"].agg(["mean", "std", "count"])
sd.index = sd.index.astype(str)
sd["ci"] = 1.96 * sd["std"] / np.sqrt(sd["count"])
sd = sd.sort_values("mean")
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.errorbar(sd["mean"], range(len(sd)), xerr=sd["ci"], fmt="o", color=BLUE, ecolor=GREY, capsize=3)
ax.axvline(sat["Satisfaction Level"].mean(), color=RED, ls="--", lw=1, label="Hospital average")
ax.set_yticks(range(len(sd))); ax.set_yticklabels(sd.index)
ax.set_title("Average satisfaction by department (95% CI)"); ax.set_xlabel("Mean satisfaction (1-5)"); ax.set_ylabel("Department")
ax.legend(frameon=False)
show()
p_sat_dept = perm_anova_p(sat["Department"].astype(str), sat["Satisfaction Level"])
print(f"Permutation ANOVA p-value, satisfaction by department: {p_sat_dept:.2f}")

**Figure 9 - Satisfaction by department.** Average satisfaction on the valid 1-5 scale is 2.99. General Practice is lowest (2.85) and Gastroenterology highest (3.09), but the confidence intervals overlap and the overall test is not significant (permutation ANOVA, p = 0.08). General Practice is the best candidate to monitor because it is also the second-busiest department.

<a id="3_3_2"></a>

### 3.3.2 Satisfaction by Patient Segment

In [ ]:
segs = [("age_group", "Age group", AGE_LAB), ("Gender", "Gender", ["Female", "Male", "Other"]),
        ("Insurance Provider", "Insurance provider", ["None", "Provider A", "Provider B", "Provider C", "Provider D"]),
        ("City of Residence", "City", None)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharey=True)
p_sat_seg = {}
for ax, (col, lab, order) in zip(axes, segs):
    a = sat.groupby(col, observed=True)["Satisfaction Level"].agg(["mean", "std", "count"])
    a.index = a.index.astype(str)
    a["ci"] = 1.96 * a["std"] / np.sqrt(a["count"])
    a = a.loc[order] if order is not None else a.sort_values("mean")
    ax.errorbar(range(len(a)), a["mean"], yerr=a["ci"], fmt="o", color=BLUE, ecolor=GREY, capsize=3, ms=7)
    ax.set_xticks(range(len(a)))
    ax.set_xticklabels([i.replace("Provider ", "") if col == "Insurance Provider" else i for i in a.index], rotation=45, ha="right")
    ax.set_xlabel(lab); ax.set_ylim(2.5, 3.5); ax.set_xlim(-.5, len(a) - .5)
    ax.axhline(sat["Satisfaction Level"].mean(), color=RED, ls="--", lw=1)
    p_sat_seg[lab] = perm_anova_p(sat[col].astype(str), sat["Satisfaction Level"])
axes[0].set_ylabel("Mean satisfaction (1-5)")
fig.suptitle("Average satisfaction by patient segment (95% CI; dashed = hospital average)", fontweight="bold")
show()
print("Permutation ANOVA p-values, satisfaction by segment:", {k: round(v, 2) for k, v in p_sat_seg.items()})

**Figure 10 - Satisfaction by patient segment.** No patient segment stands out: age group (p = 0.54), gender (p = 0.64), insurance provider (p = 0.93) and city (p = 0.95) all sit within about ±0.1 of the hospital average of 2.99. Satisfaction is therefore not explained by the demographic variables in the ABT.

<a id="3_4"></a>

## 3.4 Additional Visualizations

[Back to TOC](#toc)

Extra charts that support the insights: rating distribution, cost split, patient activity, data-cleaning evidence and feature correlations.

In [ ]:
# satisfaction distribution + relation with consultation length
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vc = sat["Satisfaction Level"].value_counts(normalize=True).sort_index() * 100
axes[0].bar(vc.index.astype(int), vc.values, color=BLUE)
axes[0].set_title("Distribution of valid ratings"); axes[0].set_xlabel("Satisfaction (1-5)"); axes[0].set_ylabel("% of rated visits")
dur_bins = pd.cut(sat["Consultation Duration"], [14, 30, 45, 60, 75, 90, 105, 120])
db = sat.groupby(dur_bins, observed=True)["Satisfaction Level"].mean()
axes[1].plot(range(len(db)), db.values, marker="o", color=BLUE, lw=2)
axes[1].set_xticks(range(len(db))); axes[1].set_xticklabels([f"{int(i.left)+1}-{int(i.right)}" for i in db.index], rotation=30)
axes[1].set_ylim(1, 5); axes[1].set_title("Satisfaction vs consultation length"); axes[1].set_xlabel("Duration (min)"); axes[1].set_ylabel("Mean satisfaction (1-5)")
show()
print(f"Correlation duration vs rating: {sat['Consultation Duration'].corr(sat['Satisfaction Level']):.2f} | "
      f"visits with invalid rating: {df['flag_rating_invalid'].mean():.1%}")

**Figure 11 - Ratings and consultation length.** Ratings are spread evenly over 1-5 and are unrelated to consultation length (correlation 0.00). Note that 7.2% of visits carried an invalid rating of 6, all but 4 of them in General Practice and ENT.

In [ ]:
# financial: who pays - insurer vs patient
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
fin = df.groupby("Insurance Provider", observed=True)[["Consultation Price", "Insurance Coverage", "out_of_pocket"]].sum()
fin.index = fin.index.astype(str)
fin = fin.loc[["None", "Provider C", "Provider A", "Provider B", "Provider D"]]
cov_pct = fin["Insurance Coverage"] / fin["Consultation Price"] * 100
axes[0].bar(range(5), cov_pct.values, color=BLUE)
axes[0].bar(range(5), 100 - cov_pct.values, bottom=cov_pct.values, color=ORANGE)
axes[0].set_xticks(range(5)); axes[0].set_xticklabels(fin.index, rotation=20)
for i, v in enumerate(cov_pct.values):
    axes[0].text(i, v / 2 if v > 15 else v + 4, f"{v:.0f}%", ha="center", color="white" if v > 15 else "black")
axes[0].set_title("Who pays: insurer (blue) vs patient (orange)"); axes[0].set_xlabel("Insurance provider"); axes[0].set_ylabel("% of consultation price")
fd = df.groupby("Department", observed=True)[["Consultation Price", "Insurance Coverage", "out_of_pocket"]].sum()
fd.index = fd.index.astype(str)
fd["avg_oop"] = df.groupby("Department", observed=True)["out_of_pocket"].mean().values
fd["avg_cov"] = df.groupby("Department", observed=True)["Insurance Coverage"].mean().values
fd = fd.sort_values("avg_oop")
axes[1].barh(fd.index, fd["avg_cov"], color=BLUE, label="Insurer pays")
axes[1].barh(fd.index, fd["avg_oop"], left=fd["avg_cov"], color=ORANGE, label="Patient pays")
axes[1].set_title("Average price per visit and who pays it"); axes[1].set_xlabel("GBP per consultation"); axes[1].set_ylabel("Department")
axes[1].legend(frameon=False, loc="lower right")
show()


**Figure 12 - Insurance coverage and out-of-pocket cost.** Insurers cover 69.9% of the £1,651,138 billed (patients pay £496,679, 30.1%). Coverage follows a fixed rule per provider (C 70%, A 80%, B 90%, D 100%, none 0%). The 90 uninsured patients (18%) pay 60% of all out-of-pocket money (≈£164 per visit). Emergency (≈£300/visit) is the costliest department; ENT and General Practice (≈£75) the cheapest.

In [ ]:
# visits per patient + recency
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(abt["n_visits"], bins=range(8, 42, 2), color=BLUE, edgecolor="white")
axes[0].set_title("Visits per patient (6 months)"); axes[0].set_xlabel("Number of visits"); axes[0].set_ylabel("Number of patients")
axes[1].hist(abt["days_since_last_visit"], bins=range(0, 40, 2), color=BLUE, edgecolor="white")
axes[1].set_title("Recency: days since last visit (at 1 Jul 2024)"); axes[1].set_xlabel("Days since last visit"); axes[1].set_ylabel("Number of patients")
show()

**Figure 13 - Visits per patient and recency.** Patients visit 20 times on average in six months (range 9-39) and the median patient was last seen only 6.5 days before the reference date (max 53). All 500 patients are active, so recency cannot identify churn in this window.

In [ ]:
# visits per patient by age group
fig, ax = plt.subplots(figsize=(7, 4))
vp = abt.groupby("age_group")["n_visits"].agg(["mean", "std", "count"]).loc[AGE_LAB]
ax.bar(range(6), vp["mean"], yerr=1.96 * vp["std"] / np.sqrt(vp["count"]), color=BLUE, capsize=3, ecolor=GREY)
ax.set_xticks(range(6)); ax.set_xticklabels(AGE_LAB); ax.set_title("Average visits per patient by age group")
ax.set_xlabel("Age group (years)"); ax.set_ylabel("Mean number of visits")
show()
print(f"ANOVA (permutation) p-value, visits by age group: {perm_anova_p(abt['age_group'], abt['n_visits']):.2f}")

**Figure 14 - Visits by age group.** The number of visits per patient does not depend on age (all groups 18.7-20.7 visits; permutation ANOVA p = 0.13).

In [ ]:
# data-cleaning evidence: raw vs clean price / income / duration
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
pairs = [("Consultation Price", "Consultation price", "GBP (log scale)", "log"),
         ("Approximate Annual Income", "Annual income", "GBP (symlog scale)", "symlog"),
         ("Consultation Duration", "Consultation duration", "Minutes", None)]
for ax, (c, title, ylab, scale) in zip(axes, pairs):
    sns.boxplot(data=[data[c].dropna().astype(float), df[c].dropna().astype(float)], ax=ax, color="#bcd3ee")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Raw", "Clean"])
    if scale == "log":
        ax.set_yscale("log")
    elif scale == "symlog":
        ax.set_yscale("symlog", linthresh=1000)
    ax.set_title(title); ax.set_ylabel(ylab); ax.set_xlabel("Dataset")
show()

**Figure 15 - Cleaning: before vs after.** Before/after view of the three numeric outliers fixed in cleaning: price (×100 errors up to £40,000), income (×10/×100 errors up to £12M) and duration (up to 600 min). After cleaning, all values lie in plausible ranges (price £50-400, duration 15-120 min).

In [ ]:
# correlation of key ABT features
cols = ["age", "annual_income", "n_visits", "n_distinct_departments", "avg_duration_min", "avg_price",
        "total_out_of_pocket", "coverage_share", "avg_satisfaction", "share_low_ratings", "visits_per_month", "days_since_last_visit"]
cm = abt[cols].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt=".1f", cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 7}, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.grid(False); ax.set_title("Correlation between key patient features"); ax.set_xlabel("Feature"); ax.set_ylabel("Feature")
show()

**Figure 16 - Correlation of key ABT features.** Strongest relations are mechanical: visits and visits per month (0.97), coverage share vs out-of-pocket (-0.95), average satisfaction vs share of low ratings (-0.86). Age-income (0.63) reflects zero income for children/students. Most behavioural and satisfaction features are nearly uncorrelated, so a predictive model should not expect strong signals.

<a id="3_5"></a>

## 3.5 Management Questions and Key Insights

[Back to TOC](#toc)

In [ ]:
# Key figures behind the answers below (all computed from the cleaned data)
days = mt.index.days_in_month
print("Q1  Visits per department (share of all visits):")
display(pd.DataFrame({"visits": dv, "share_%": (dv / dv.sum() * 100).round(1)}).T)
print("    Visits per month and per day:")
display(pd.DataFrame({"visits": mt.values, "per_day": (mt.values / days).round(1)}, index=[m.strftime("%b") for m in mt.index]).T)
print("    Change in visits, Jan -> Jun, by department:")
display((pm.loc[pm.index.max()] - pm.loc[pm.index.min()]).sort_values().to_frame("jun_minus_jan").T)

print("Q2  Most used department by age group and gender (share of that group's visits):")
display(pd.DataFrame({"top_department": da.idxmax(axis=1), "share_%": da.max(axis=1).round(1)}).T)
dgx = pd.crosstab(df["Gender"].astype(str), df["Department"].astype(str), normalize="index") * 100   # share of each gender's visits
print("    ... and by gender:")
display(pd.DataFrame({"top_department": dgx.idxmax(axis=1), "share_%": dgx.max(axis=1).round(1)}).T)
print(f"    chi-square p-values - department vs age group: {p_dept_age:.2f} | vs gender: {p_dept_gender:.2f}")

print("Q3  Satisfaction (valid ratings), lowest departments and overall:")
display(sd[["mean", "count"]].round(2).head(4).T)
print(f"    overall {sat['Satisfaction Level'].mean():.2f} | ANOVA by department p = {p_sat_dept:.2f}")
print("    segment tests (p-values):", {k: round(v, 2) for k, v in p_sat_seg.items()})

print("Q4  Who pays the consultation price:")
tot_price, tot_cov = df["Consultation Price"].sum(), df["Insurance Coverage"].sum()
tot_oop = tot_price - tot_cov
n_uninsured = int((abt["has_insurance"] == 0).sum())
oop_uninsured = df.loc[df["Insurance Provider"].astype(str) == "None", "out_of_pocket"].sum() / tot_oop
print(f"    billed £{tot_price:,.0f} | insurers £{tot_cov:,.0f} ({tot_cov / tot_price:.1%}) | patients £{tot_oop:,.0f} ({tot_oop / tot_price:.1%})")
print(f"    uninsured patients: {n_uninsured} ({n_uninsured / 5:.0f}%), they pay {oop_uninsured:.0%} of all out-of-pocket money")
print(f"    average price per visit £{df['Consultation Price'].mean():.0f}, average out-of-pocket £{df['out_of_pocket'].mean():.0f}")

**Answers to management's questions**

1. **Busiest departments and demand from January to June.** Psychiatry (1,359 visits) and General Practice (1,331) are the busiest, together 27% of all consultations and about twice any other department; Orthopedics is the quietest (593). Total demand is flat over the six months (1,728 visits in January, 1,616 in June; ~53-57 visits per day in every month). Departments fluctuate month to month (Psychiatry 254 → 197) but there is no consistent trend or seasonality.
2. **Which patient groups use which departments.** Department mix is very similar across age groups (p = 0.20) and genders (p = 0.23), so no group is tied to a department. The most-used department is General Practice for children (0-17) and for 65+, and Psychiatry for ages 18-64, but always with only ~14% of the group's visits, a negligible lead. Only weak hints exist (Orthopedics lowest for 50-64, Emergency lowest and Pulmonology highest for 80+) and should be treated as hypotheses.
3. **Lowest satisfaction.** Average satisfaction is 2.99/5. General Practice is lowest (2.85) and Gastroenterology highest (3.09), but the differences are not significant (p = 0.08). No patient segment (age, gender, insurer, city) differs from the average either, so satisfaction is not explained by the demographics in the ABT. General Practice deserves monitoring because it is both the lowest-rated and the second-busiest department.
4. **Insurance vs out-of-pocket.** Insurers cover 69.9% of the £1.65M billed; patients pay £496,679 (30.1%). Coverage is a fixed rate per provider (C 70%, A 80%, B 90%, D 100%, none 0%). The 90 uninsured patients (18%) pay about 60% of all out-of-pocket money (≈£164 per visit).

**Suggestions for the next analytics team**

- **Sensible targets:** `avg_satisfaction` / `share_low_ratings` (experience), `n_visits` / `visits_per_month` (utilisation), `total_out_of_pocket` or `oop_to_income_pct` (financial burden) and `top_department` (service demand). Expect weak signals: most behavioural and satisfaction features are almost uncorrelated.
- **Segmentation ideas:** cluster on age group, income, insurance status and utilisation (`n_visits`, `n_distinct_departments`, `share_emergency`); treat uninsured patients as a distinct group because of their financial exposure.
- **Caveats:** all patients were active in the window, so recency cannot define churn; the `Other` gender share (~1/3) should be confirmed; the rating-6 code is concentrated in two departments; four patients have a profession that does not fit their age (`profession_age_odd`); and visits flagged in `flag_same_day_same_dept` may need review. Use the `dq_*` columns to test how sensitive any result is to repaired data.

## Summary
See the accompanying report and slides (PDF) for the full discussion, the data dictionary and recommendations for the next analytics team.